In [1]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection/

fatal: destination path 'Continual-hate-speech-detection' already exists and is not an empty directory.
/content/Continual-hate-speech-detection


In [2]:
from dataset.df_loader import getdf_davidson, getdf_hatexplain

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print(len(df_dv), len(df_hx))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                                                text      label    source
0  rt as a woman you shouldnt complain about clea...     normal  davidson
1  rt boy dats coldtyga dwn bad for cuffin dat ho...  offensive  davidson
2  rt dawg rt you ever fuck a bitch and she start...  offensive  davidson
3                          rt she look like a tranny  offensive  davidson
4  rt the shit you hear about me might be true or...  offensive  davidson
                                                text       label      source
0  i dont think im getting my baby them white 9 h...      normal  hatexplain
1  we cannot continue calling ourselves feminists...      normal  hatexplain
2                      nawt yall niggers ignoring me      normal  hatexplain
3  user i am bit confused coz chinese ppl can not...  hatespeech  hatexplain
4  this bitch in whataburger eating a burger with...  hatespeech  hatexplain
24275 20144


In [3]:
import torch

from transformers import (
    AutoConfig,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorWithPadding
)

from models.model_builder import CustomClassifier

from models.trainer import (
    build_trainer,
    train_model,
    create_run_dir,
    save_run
)

from dataset.df_loader import (
    getdf_davidson,
    getdf_hatexplain
)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

cpu


In [5]:
import torchao
print(f"torchao version: {torchao.__version__}")

torchao version: 0.17.0


In [6]:
num_labels = 2

config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

model = CustomClassifier(
    model_name="roberta-base",
    config=config,
    class_weights=None,
    use_lora=True
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
training_args = TrainingArguments(
    output_dir="outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50
)

In [9]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
trainer = build_trainer(
    model=model,
    training_args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

result = train_model(trainer)

In [ ]:
config_dict = {
    "model_name": "roberta-base",
    "num_labels": 2,
    "learning_rate": 2e-5,
    "batch_size": 16,
    "epochs": 3,
    "use_lora": True
}

run_dir = create_run_dir()

save_run(
    trainer,
    tokenizer,
    config_dict,
    run_dir
)